## Dependencies

In [ ]:
## libraries
import sys
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import (
    _to_results_frame,
    logo_cross_valid,
    kfold_cross_valid
)
from src.evaluators.metrics import FRONTIER_METRICS
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initalization

In [2]:
## setup
data = load_processed_data()
models = load_estimators()

## Training

In [3]:
## leave-one-group-out cross validation (domain)
results_dict_domain = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain",  ## fold on domain
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_domain[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [4]:
## leave-one-group-out cross validation (discipline)
results_dict_disc = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "discipline",  ## fold on discipline
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_disc[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [5]:
## 2-fold cross validation
results_dict_2fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 2,  ## fold on random 50-50 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_2fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [6]:
## 5-fold cross validation
results_dict_5fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 5,  ## fold on random 80-20 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_5fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [7]:
## 10-fold cross validation
results_dict_10fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 10,  ## fold on random 90-10 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_10fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


## Post-Processing

In [ ]:
## convert all results to dataframes
results_data_domain = _to_results_frame(results_dict_domain)
results_data_disc = _to_results_frame(results_dict_disc)
results_data_2fold = _to_results_frame(results_dict_2fold)
results_data_5fold = _to_results_frame(results_dict_5fold)
results_data_10fold = _to_results_frame(results_dict_10fold)

## Results

In [9]:
## generalizability table: mean frontier metrics by cv protocol
_protocols = {
    "LOGO-CV Domain":     results_data_domain,
    "LOGO-CV Discipline": results_data_disc,
    "2-Fold-CV (m=30)":   results_data_2fold,
    "5-Fold-CV (m=30)":   results_data_5fold,
    "10-Fold-CV (m=30)":  results_data_10fold,
}

def _protocol_summary(frame: pd.DataFrame) -> pd.Series:
    ## logo outputs have one row per held-out group; collapse to one row per learner
    if "group" in frame.columns:
        frame = frame.groupby("model", observed = True)[FRONTIER_METRICS].mean()
    else:
        frame = frame.set_index("model")[FRONTIER_METRICS]

    ## one value per learner, then average across learners
    return frame.mean()

_rows = {
    name: _protocol_summary(frame)
    for name, frame in _protocols.items()
}

results_table_transfer = pd.DataFrame(_rows).T.round(4)
results_table_transfer.index.name = "Protocol"
results_table_transfer.columns = [c.upper() for c in results_table_transfer.columns]
results_table_transfer.loc["Mean"] = results_table_transfer.mean().round(4)

display(results_table_transfer)

,VR,MV,MS,EA,EI
Protocol,,,,,
LOGO-CV Domain,0.0873,3.8293,5.5200,0.9318,0.7242
LOGO-CV Discipline,0.1046,1.1335,8.1668,1.7535,0.7079
2-Fold-CV (m=30),0.0820,0.9158,4.7526,0.7060,0.7094
5-Fold-CV (m=30),0.0744,5.1004,7.6744,1.1677,0.7401
10-Fold-CV (m=30),0.0954,15.3702,9.3050,1.5186,0.7363
Mean,0.0887,5.2698,7.0838,1.2155,0.7236
